[![Open In Colab](https://google.com)](https://google.comhamagami/IS2026/blob/main/Demo_31_MCTS_LearnedModel.ipynb)

# Demo 31: Tree Search: Known vs Learned Model

**挿入位置：AlphaZero・MuZeroの比較スライド**　｜　**実行環境：Google Colab**

**概念デモ**：2手先までの分岐木を使い、（A）与えられた報酬モデルで探索する場合と、（B）経験から得た近似報酬モデルで探索する場合を比較します。AlphaZero/MuZeroの大規模なニューラルネットワーク実装ではありません。

**使い方**：ランタイムで「すべてのセルを実行」した後、スライダー／ボタンで条件を変更します。グラフ中の説明は英語、手順と解説は日本語です。CPUで動作します（Demo 28/29 はPyTorchを使用）。

**報酬の添字**：状態 $s_t$ で行動 $a_t$ を実行した結果得る報酬を $r_t$ と呼び、次状態を $s_{t+1}$ とします。経験は $(s_t,a_t,r_t,s_{t+1})$ です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
from collections import defaultdict
ACTIONS_M=['Left','Center','Right']
# Hidden oracle task rewards, depending on the sequence of 2 actions.
TRUE=np.array([[.35,.85,.22],[.52,.41,.72],[.18,.65,1.05]])
NOISE=.1

def collect_experience(n=50,seed=0):
    rng=np.random.default_rng(seed);a=rng.integers(0,3,size=n);b=rng.integers(0,3,size=n)
    r=TRUE[a,b]+rng.normal(0,NOISE,n)
    return a,b,r

def learn_reward_model(n=50,seed=0,prior=.1):
    aa,bb,rr=collect_experience(n,seed);count=np.zeros((3,3));summ=np.zeros((3,3))
    for a,b,r in zip(aa,bb,rr):count[a,b]+=1;summ[a,b]+=r
    # Smooth unseen action sequences to neutral prior, not true reward.
    return (summ+prior*.5)/(count+prior),count

class SearchNode:
    def __init__(self,seq=()):self.seq=seq;self.visits=0;self.sum_reward=0.;self.children={}

def mcts(reward_table,budget=100,seed=0):
    rng=np.random.default_rng(seed);root=SearchNode(())
    for _ in range(budget):
        node=root;path=[node]
        while len(node.seq)<2:
            if len(node.children)<3:
                unseen=[a for a in range(3) if a not in node.children]
                a=int(rng.choice(unseen));node.children[a]=SearchNode(node.seq+(a,));node=node.children[a];path.append(node)
                break
            # UCB1 to balance exploring unknown branches and using known values.
            a=max(node.children,key=lambda a:(node.children[a].sum_reward/max(1,node.children[a].visits)+1.2*np.sqrt(np.log(node.visits+1)/max(1,node.children[a].visits))))
            node=node.children[a];path.append(node)
        seq=node.seq
        while len(seq)<2:seq=seq+(int(rng.integers(3)),)
        outcome=float(reward_table[seq[0],seq[1]])
        for node in path:node.visits+=1;node.sum_reward+=outcome
    stats={a:(root.children[a].visits,root.children[a].sum_reward/max(1,root.children[a].visits)) if a in root.children else (0,0.) for a in range(3)}
    selected=max(stats,key=lambda a:stats[a][0])
    return selected,stats,root

m,counts=learn_reward_model(100,0);pick,stats,root=mcts(m,100,0)
assert sum(v[0] for v in stats.values())==100

In [ ]:
experiences=widgets.IntSlider(value=40,min=5,max=250,step=5,description='Experiences',continuous_update=False)
budget=widgets.IntSlider(value=90,min=9,max=500,step=10,description='MCTS sims',continuous_update=False)
seed=widgets.IntSlider(value=4,min=0,max=100,description='Seed')
model=widgets.ToggleButtons(options=['Known model (oracle)','Learned model (from data)'],description='Planning')
button=widgets.Button(description='Run tree search',button_style='primary');out=widgets.Output()
def draw_search(_):
    with out:
        clear_output(wait=True)
        learned,cnt=learn_reward_model(experiences.value,seed.value)
        estimates=TRUE if model.value.startswith('Known') else learned
        chosen,stats,tree=mcts(estimates,budget.value,seed.value)
        fig,ax=plt.subplots(1,3,figsize=(14.5,3.9))
        ax[0].imshow(TRUE,vmin=0,vmax=1.15,cmap='YlGnBu');ax[0].set(title='True rewards (shown for teaching)',xticks=range(3),yticks=range(3),xticklabels=ACTIONS_M,yticklabels=ACTIONS_M,xlabel='Second action',ylabel='First action')
        for i in range(3):
            for j in range(3):ax[0].text(j,i,f'{TRUE[i,j]:.2f}',ha='center',va='center')
        ax[1].imshow(learned,vmin=0,vmax=1.15,cmap='YlGnBu');ax[1].set(title=f'Learned rewards ({experiences.value} samples)',xticks=range(3),yticks=range(3),xticklabels=ACTIONS_M,yticklabels=ACTIONS_M,xlabel='Second action',ylabel='First action')
        for i in range(3):
            for j in range(3):ax[1].text(j,i,f'{learned[i,j]:.2f}',ha='center',va='center')
        visits=[stats[i][0] for i in range(3)];ax[2].bar(ACTIONS_M,visits,color=['#9ACAF4' if i!=chosen else '#F8B6CA' for i in range(3)]);ax[2].set(title='MCTS root visit counts',ylabel='Visits',xlabel='First action')
        plt.tight_layout();plt.show()
        print('Selected first action:',ACTIONS_M[chosen])
        print('Visit count / simulated mean return:',{ACTIONS_M[i]:(stats[i][0],round(stats[i][1],3)) for i in range(3)})
        print('True best first action (best second action thereafter):',ACTIONS_M[int(np.argmax(TRUE.max(axis=1)))])
        if model.value.startswith('Learned'):print('Sequences observed:',int((cnt>0).sum()),'/ 9; predicted model is not guaranteed accurate.')
button.on_click(draw_search)
display(widgets.VBox([widgets.HBox([experiences,budget,seed]),model,button,out]));draw_search(None)

### AlphaZeroとMuZeroとの対応（重要な注意）
- **AlphaZeroの考え方**：ゲームのルール（状態遷移）を与え、自己対戦によって方策・価値を学び、木探索を使用します。
- **MuZeroの考え方**：外部から完全な状態遷移モデルを与えず、経験から**意思決定に必要な潜在的ダイナミクス・報酬・価値・方策**を学び、そのモデルで木探索を行います。
- この簡略デモで学ぶのは、**「探索に使う予測モデルが既知か、経験から学習したものか」**という対比です。ここでは画像の潜在状態表現や本物のMuZeroネットワークを実装していません。
- 学習モデルが不十分なら、MCTSの探索回数だけを増やしても正しい判断とは限りません。

## Try it yourself / 実験課題
- Experiencesを10から150に増やして、学習モデルの推定値がどう変わるかを観察する。
- MCTS simsを増やし、選ばれる初手が変わるか確認する。
- 「モデルの誤り」と「探索不足」という2種類の問題を区別する。